# Part 8 - Ensemble Training & Blending
Train tuned models, create OOF predictions, optimize blend weights, and generate the final submission.

In [ ]:
import json
import numpy as np
import pandas as pd

from itertools import product
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

TARGET='addicted_label'
ID='id'

with open('best_params.json') as f:
    BEST=json.load(f)

FEATURES=[c for c in train_fe.columns if c not in [ID,TARGET]]

X=train_fe[FEATURES].copy()
y=train_fe[TARGET]
X_test=test_fe[FEATURES].copy()
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)


## Prepare datasets

In [ ]:
cat_cols=X.select_dtypes(include=['object','category']).columns.tolist()

# CatBoost
X_cat=X.copy()
X_test_cat=X_test.copy()
cat_idx=[X_cat.columns.get_loc(c) for c in cat_cols]

# LightGBM
X_lgb=X.copy()
X_test_lgb=X_test.copy()
for c in cat_cols:
    X_lgb[c]=X_lgb[c].astype('category')
    X_test_lgb[c]=X_test_lgb[c].astype('category')

# XGBoost
X_xgb=X.copy()
X_test_xgb=X_test.copy()
enc=OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)
if len(cat_cols):
    X_xgb[cat_cols]=enc.fit_transform(X_xgb[cat_cols].astype(str))
    X_test_xgb[cat_cols]=enc.transform(X_test_xgb[cat_cols].astype(str))


## Cross-validation

In [ ]:
oof_cat=np.zeros(len(X))
oof_lgb=np.zeros(len(X))
oof_xgb=np.zeros(len(X))

pred_cat=np.zeros(len(X_test))
pred_lgb=np.zeros(len(X_test))
pred_xgb=np.zeros(len(X_test))

for fold,(tr,va) in enumerate(cv.split(X,y),1):

    print(f"Fold {fold}")

    cat=CatBoostClassifier(
        **BEST['catboost'],
        iterations=3000,
        loss_function='Logloss',
        eval_metric='AUC',
        verbose=False,
        random_seed=42
    )
    cat.fit(X_cat.iloc[tr],y.iloc[tr],
            cat_features=cat_idx,
            eval_set=(X_cat.iloc[va],y.iloc[va]),
            use_best_model=True)

    oof_cat[va]=cat.predict_proba(X_cat.iloc[va])[:,1]
    pred_cat+=cat.predict_proba(X_test_cat)[:,1]/5

    lgb=LGBMClassifier(
        **BEST['lightgbm'],
        objective='binary',
        metric='auc',
        n_estimators=3000,
        random_state=42
    )
    lgb.fit(X_lgb.iloc[tr],y.iloc[tr],categorical_feature=cat_cols)
    oof_lgb[va]=lgb.predict_proba(X_lgb.iloc[va])[:,1]
    pred_lgb+=lgb.predict_proba(X_test_lgb)[:,1]/5

    xgb=XGBClassifier(
        **BEST['xgboost'],
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist',
        n_estimators=3000,
        random_state=42
    )
    xgb.fit(X_xgb.iloc[tr],y.iloc[tr],verbose=False)
    oof_xgb[va]=xgb.predict_proba(X_xgb.iloc[va])[:,1]
    pred_xgb+=xgb.predict_proba(X_test_xgb)[:,1]/5


## Optimize Blend Weights

In [ ]:
best_auc=0
best_w=None

for wc,wl in product(np.arange(0,1.05,0.05),repeat=2):
    wx=1-wc-wl
    if wx<0:
        continue
    blend=wc*oof_cat+wl*oof_lgb+wx*oof_xgb
    auc=roc_auc_score(y,blend)
    if auc>best_auc:
        best_auc=auc
        best_w=(wc,wl,wx)

print("Best AUC:",best_auc)
print("Weights:",best_w)


## Final Submission

In [ ]:
wc,wl,wx=best_w
final_pred=wc*pred_cat+wl*pred_lgb+wx*pred_xgb

submission=sample.copy()
submission[TARGET]=final_pred
submission.to_csv("submission_ensemble.csv",index=False)

display(submission.head())
print("Saved submission_ensemble.csv")


### Next
Part 9 will add SHAP explainability, error analysis, calibration, threshold experiments, and final diagnostics.